# 01 Dataset exploration (IDD Detection)
Run after `scripts/prepare_dataset.py`. All numbers come from the converted dataset; nothing is typed in by hand.

This notebook answers: how many images and boxes are there, which classes are rare, how big are the objects, and did the conversion drop anything?

## Setup
Load the configuration, the class list and every converted label into one table. `sys.path.insert` lets the notebook import the repository's `src/` package, because the notebook lives in `notebooks/` (one folder below the repository root).

In [ ]:
import sys, json
from pathlib import Path

# Make `src` importable: the notebook runs inside notebooks/, so the repo root is one level up.
sys.path.insert(0, str(Path.cwd().parent))
import pandas as pd
from IPython.display import Image, display
from src.data.config import load_classes, load_yaml, resolve
from src.visualization.eda import load_boxes, compute_stats, run_eda

# Read the dataset configuration and the 15-class list (same files the scripts use).
cfg = load_yaml(resolve('configs/dataset.yaml'))
classes = load_classes(resolve(cfg['class_mapping']))
names = [c.name for c in classes]               # class names in id order
groups = {c.name: c.group for c in classes}     # class name -> common / ambiguous / india_specific
yolo = resolve(cfg['yolo']['root'])             # folder with the converted YOLO dataset

# manifest.csv has one row per converted image (split, name, size, number of boxes).
manifest = pd.read_csv(yolo / 'manifest.csv')
# load_boxes reads every label file into one table with one row per box.
boxes = load_boxes(yolo, names)
# The last expression of a cell is displayed: here, the number of images per split.
manifest.groupby('split').size()

## Conversion report and issues
Every box or image the converter dropped, clipped or flagged is logged with a reason code. Reading the log here shows how clean the data was and that nothing disappeared silently.

In [ ]:
# Per-split counts written by the converter (images converted, boxes kept/dropped, ...).
print(json.dumps(json.load(open(yolo / 'conversion_report.json')), indent=2))

# One row per logged problem; count them by severity (error/warning) and reason code.
issues = pd.read_csv(yolo / 'conversion_issues.csv')
issues.groupby(['severity', 'code']).size()

## Class distribution
Number of boxes per class and split. A very uneven table here is the class imbalance that explains why rare classes (caravan, trailer, train) are hard to learn and to measure.

In [ ]:
# Count boxes per (class, split), turn splits into columns, keep all classes even if a count is 0.
counts = boxes.groupby(['class', 'split']).size().unstack(fill_value=0).reindex(names, fill_value=0)
# Add each class's group so common vs India-specific classes can be compared at a glance.
counts['group'] = counts.index.map(groups)
counts

## Statistics and figures (written to results/eda/)
`run_eda` is the same function `scripts/analyze_dataset.py` calls: it computes all statistics, saves five figures plus `eda_stats.json` and `class_counts.csv`, and returns the stats.

In [ ]:
out = resolve('results/eda')
# Compute the statistics and (re)draw every figure into results/eda/.
stats = run_eda(yolo, names, groups, out, cfg['release']['name'])
# Show each saved figure inside the notebook.
for f in stats['figures']:
    display(Image(filename=str(out / f)))

## Observations
Write these after looking at the real figures (imbalance, rare classes, small objects, train vs val differences).